In [1]:
import pandas as pd
import numpy as np
from datetime import datetime
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.ensemble import GradientBoostingRegressor,RandomForestRegressor
from sklearn.preprocessing import LabelEncoder,MinMaxScaler
from sklearn.linear_model import Ridge

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense

In [2]:
# Loading dataset for initial inspections
df = pd.read_csv("agro_moisture_dataset.csv")

df.head() 
df.shape

(913, 10)

In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 913 entries, 0 to 912
Data columns (total 10 columns):
 #   Column                         Non-Null Count  Dtype 
---  ------                         --------------  ----- 
 0   Sensor_ID                      913 non-null    object
 1   Date                           909 non-null    object
 2   Soil_Moisture(%)               913 non-null    object
 3   Soil_pH                        913 non-null    object
 4   Temperature(C)                 913 non-null    object
 5   Humidity(%)                    913 non-null    object
 6   Crop_Type                      909 non-null    object
 7   Fertilizer_Recommended(kg/ha)  913 non-null    object
 8   Irrigation_Recommended(mm)     913 non-null    object
 9   Drone_Image_ID                 913 non-null    object
dtypes: object(10)
memory usage: 71.5+ KB


In [4]:
#Descriptive Statistics
df.describe()

,Sensor_ID,Date,Soil_Moisture(%),Soil_pH,Temperature(C),Humidity(%),Crop_Type,Fertilizer_Recommended(kg/ha),Irrigation_Recommended(mm),Drone_Image_ID
count,913,909,913,913,913,913,909,913,913,913
unique,800,798,750,313,170,448,5,595,243,800
top,SEN-1430,2026-09-10,error,7.54,19.1,56.0,Wheat,66.9,8.0,IMG-2430
freq,2,2,5,11,13,8,203,8,11,2


In [5]:
#Cleaning Data

#Checking missing values
print("------MISSING VALUES------")
df.isnull().sum()

------MISSING VALUES------


Sensor_ID                        0
Date                             4
Soil_Moisture(%)                 0
Soil_pH                          0
Temperature(C)                   0
Humidity(%)                      0
Crop_Type                        4
Fertilizer_Recommended(kg/ha)    0
Irrigation_Recommended(mm)       0
Drone_Image_ID                   0
dtype: int64

In [6]:

#Handling invalid values and missing values

#Numerical columns are filled with median or mean
numeric_cols = ['Soil_Moisture(%)','Soil_pH','Temperature(C)','Humidity(%)','Fertilizer_Recommended(kg/ha)','Irrigation_Recommended(mm)']

for col in numeric_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce') #changing invalid values to NaN
        x = df[col].median()
        df[col] = df[col].fillna(x) #filling NaN with median
    
        
 #categorical values are filled with mode   
categorical_cols = ['Crop_Type'] 
for col in categorical_cols:
    if col in df.columns:
        df[col] = df[col].fillna(df[col].mode()[0])
        
#Changing Date Format
df['Date'] = pd.to_datetime(df['Date'], errors = 'coerce')
df['Date'] = df['Date'].bfill()
df = df.sort_values(by="Date")
        
#checking for missing values again
print("------MISSING VALUES------")
df.isnull().sum()



------MISSING VALUES------


Sensor_ID                        0
Date                             0
Soil_Moisture(%)                 0
Soil_pH                          0
Temperature(C)                   0
Humidity(%)                      0
Crop_Type                        0
Fertilizer_Recommended(kg/ha)    0
Irrigation_Recommended(mm)       0
Drone_Image_ID                   0
dtype: int64

In [7]:
#Handling Wrong data

factor_cols = ['Soil_Moisture(%)','Soil_pH','Temperature(C)','Humidity(%)','Fertilizer_Recommended(kg/ha)','Irrigation_Recommended(mm)']
for col in factor_cols:
    if col in df.columns:
        mean = df[col].mean()
        std = df[col].std()
        df[col] = np.where((df[col] < mean - 3*std) | (df[col] > mean + 3*std), np.nan, df[col])
        df[col] = df[col].fillna(df[col].median())

In [8]:
#checking for duplicates
print("------DUPLICATED ROWS------")
print(df.duplicated().sum())

------DUPLICATED ROWS------
111


In [9]:
#removing duplicates
df = df.drop_duplicates()
print("number of rows and columns after removing duplicates:",df.shape)

#checking Date duplicates
print("Date duplicates:",df['Date'].duplicated().sum())

#drop the duplicates
df = df.drop_duplicates(subset = ['Date'], keep = 'first')

print('Shape:',df.shape)


number of rows and columns after removing duplicates: (802, 10)
Date duplicates: 4
Shape: (798, 10)


In [10]:
df.info()
print("-------------------------------------")
print("Missing values after cleaning")
print(df.isnull().sum())
df.to_csv("cleaned_agro_moisture_dataset.csv")

<class 'pandas.core.frame.DataFrame'>
Index: 798 entries, 0 to 799
Data columns (total 10 columns):
 #   Column                         Non-Null Count  Dtype         
---  ------                         --------------  -----         
 0   Sensor_ID                      798 non-null    object        
 1   Date                           798 non-null    datetime64[ns]
 2   Soil_Moisture(%)               798 non-null    float64       
 3   Soil_pH                        798 non-null    float64       
 4   Temperature(C)                 798 non-null    float64       
 5   Humidity(%)                    798 non-null    float64       
 6   Crop_Type                      798 non-null    object        
 7   Fertilizer_Recommended(kg/ha)  798 non-null    float64       
 8   Irrigation_Recommended(mm)     798 non-null    float64       
 9   Drone_Image_ID                 798 non-null    object        
dtypes: datetime64[ns](1), float64(6), object(3)
memory usage: 68.6+ KB
------------------------

In [11]:
#VISUALIZATIONS
'''
#1.SOIL MOISTURE LINE GRAPH
plt.figure(figsize=(10,5))
sns.lineplot(x=df['Date'], y=df['Soil_Moisture(%)'])
plt.title('Soil Moisture Trend Over Time')
plt.xlabel('Date')
plt.ylabel('Soil Moisture(%)')
plt.show()

#2.SOIL MOISTURE HISTOGRAM
plt.figure(figsize=(10,5))
sns.histplot(df['Soil_Moisture(%)'],bins=30, kde=True)
plt.title('Soil Moisture Histogram')
plt.show()

#3.MOISTURE VS TEMPERATURE SCATTER PLOT
plt.figure(figsize=(10,5))
sns.scatterplot(x=df['Soil_Moisture(%)'], y=df['Temperature(C)'])
plt.title('Soil Moisture vs Temperature')
plt.xlabel('Temperature')
plt.ylabel('Soil Moisture(%)')
plt.show()

#4.BOXPLOT FOR MOISTURE AND TEMPEATURE
plt.figure(figsize=(10,5))
sns.boxplot(data=df[['Temperature(C)','Soil_Moisture(%)']])
plt.title('Box plot for Soil moisture and Temperature')
plt.show()

#5.HEATMAP 
plt.figure(figsize=(8,5))
#COMPUTE CORRELATION MATRIX
corr_matrix = df[['Soil_Moisture(%)','Soil_pH','Temperature(C)','Humidity(%)','Fertilizer_Recommended(kg/ha)','Irrigation_Recommended(mm)']].corr()
sns.heatmap(corr_matrix, annot=True, cmap="coolwarm", fmt='.2f')
plt.title("Correlation Heatmap")
plt.show()

#6.BARPLOT
plt.figure(figsize=(10,5))
sns.barplot(x='Crop_Type', y='Soil_Moisture(%)',hue='Crop_Type', data=df, palette=['red','green','orange','blue','purple'])
plt.title("Soil Moisture By Crop Type Barplot")
plt.xlabel('Crop Type')
plt.ylabel('Soil Moisture(%)')
plt.show()

#7.CROP TYPE PIECHART
plt.figure(figsize=(6,6))
#count the number of each crop type
crop_count = df['Crop_Type'].value_counts()
plt.pie(crop_count, labels=crop_count.index, colors=['green','red','orange','blue','purple'])
plt.legend(title = 'Crop Types', loc='lower left')
plt.title('Crop Type Distribution')
plt.show()

#8.PAIRPLOT
plt.figure(figsize=(10,5))
sns.pairplot(df, hue='Crop_Type', diag_kind='kde',palette='Set2') 

plt.show()

#9.DENSITYPLOT
plt.figure(figsize=(10,5))
sns.kdeplot(df['Soil_Moisture(%)'], label = 'Soil Moisture', fill=True)
sns.kdeplot(df['Temperature(C)'], label ='Temperature', fill=True)
plt.title("Soil Moisture and Temperature Density Plot")
plt.legend()
plt.ylabel('Density')
plt.show()

#10.AREA CHART
plt.figure(figsize=(10,5))
agg = df.groupby('Crop_Type')['Soil_Moisture(%)'].mean()
plt.fill_between(agg.index, agg.values,color='skyblue', alpha=0.5)
plt.plot(agg.index, agg.values,color='Slateblue', linewidth=2)
plt.title("Moisture by Crop type Area chart")
plt.ylabel("Soil Moisture")
plt.xlabel("Crop Type")
plt.xticks(rotation = 45)
plt.tight_layout()
plt.show()

#11.COUNT PLOT
plt.figure(figsize =(10,5))
sns.countplot(data=df, x='Crop_Type', palette="Set2")
plt.xlabel('Crop Type')
plt.ylabel('Numble of samples')
plt.title("Count plot for Crop Types")
plt.show()

#12.TIMESERIES PLOT
agg = df.groupby('Date')['Soil_Moisture(%)'].mean()
plt.figure(figsize=(12,6))
plt.plot(agg.index,agg.values, marker='o', color='blue')
plt.xlabel('Years')
plt.ylabel('Soil Moisture')
plt.title('Soil Moisture over Time')
plt.grid(True)
plt.show()

#13.BOX PLOT FOR FETRILIZER VS SOIL MOISTURE
plt.figure(figsize=(10,5))
sns.boxplot(data=df[['Fertilizer_Recommended(kg/ha)','Soil_Moisture(%)']])
plt.title('Box plot for Soil moisture and Recommended Fertilizer')
plt.show()

#14.HISTOGRAM FOR SOIL PH
plt.figure(figsize=(10,5))
sns.histplot(df['Soil_pH'],bins=30, kde=True)
plt.title('Soil pH Histogram')
plt.show()

#15.SCATTERPLOT FOR SOIL MOISTURE AND IRRIGATION
plt.figure(figsize=(10,5))
sns.scatterplot(x=df['Soil_Moisture(%)'], y=df['Irrigation_Recommended(mm)'])
plt.title('Soil Moisture vs Irrigation')
plt.xlabel('Irrigation')
plt.ylabel('Soil Moisture(%)')
plt.show()

#16.LINE PLOT FOR HUMIDITY
plt.figure(figsize=(10,5))
sns.lineplot(x=df['Humidity(%)'], y=df['Soil_Moisture(%)'])
plt.title('Soil Moisture Trend alongside Humidity')
plt.xlabel('Humidity')
plt.ylabel('Soil Moisture(%)')
plt.show()

#17 TEMPERATURE HISTOGRAM
plt.figure(figsize=(10,5))
sns.histplot(df['Temperature(C)'],bins=30, kde=True)
plt.title('Temperature Histogram')
plt.show()

#18.MOISTURE VS TEMPERATURE SCATTER PLOT
plt.figure(figsize=(10,5))
scatter = sns.scatterplot(x='Soil_Moisture(%)', y='Temperature(C)', hue='Humidity(%)',data=df,palette='coolwarm',alpha=0.7,sizes=(20,200))
plt.title('Soil Moisture vs Temperature by humidity')
plt.xlabel('Temperature')
plt.ylabel('Soil Moisture(%)')
plt.legend(title="Humidity", bbox_to_anchor=(1.05,1), loc='upper left')
plt.grid(True)
plt.show()

#19.VIOLIN PLOT
plt.figure(figsize=(10,5))
sns.violinplot(x = 'Crop_Type', y='Soil_Moisture(%)',data=df, inner='quartile')
plt.title("Moisture Distribution by Crop Type")
plt.show()

#20.HUMIDITY HISTOGRAM
plt.figure(figsize=(10,5))
sns.histplot(df['Humidity(%)'],bins=30, kde=True)
plt.title('Humidity Histogram')
plt.show()

#21 RADAR CHART

labels = ['Soil_Moisture(%)','Soil_pH','Temperature(C)','Humidity(%)','Fertilizer_Recommended(kg/ha)','Irrigation_Recommended(mm)']
values = [df['Soil_Moisture(%)'].mean(),
          df['Soil_pH'].mean(),
          df['Temperature(C)'].mean(),
          df['Humidity(%)'].mean(),
          df['Fertilizer_Recommended(kg/ha)'].mean(),
          df['Irrigation_Recommended(mm)'].mean()]
values += values[:1]
angles = np.linspace(0, 2*np.pi, len(labels), endpoint=False)
angles = np.concatenate((angles,[angles[0]]))
fig, ax = plt.subplots(subplot_kw={'polar':True})
ax.plot(angles,values)
ax.fill(angles,values, alpha =0.2)
ax.set_xticks(angles[:-1])
ax.set_xticklabels(labels)
plt.title("factors affecting soil moisture")
plt.show()

#22IRRIGATION HISTOGRAM
plt.figure(figsize=(10,5))
sns.histplot(df['Irrigation_Recommended(mm)'],bins=30, kde=True)
plt.title('Irrigation_Recommended(mm)')
plt.show()

#23FERTILIZER HISTOGRAM
plt.figure(figsize=(10,5))
sns.histplot(df['Fertilizer_Recommended(kg/ha)'],bins=30, kde=True)
plt.title("Histogram for Fertilizers Recommended")
plt.show()

#24 BOXPLOT FOR MPOISTURE AND HUMIDITY
plt.figure(figsize=(10,5))
sns.boxplot(data=df[['Humidity(%)','Soil_Moisture(%)']])
plt.title('Box plot for Soil moisture and Humidity')
plt.show()

#BOXPLOT FOR MOISTURE AND IRRIGATION
plt.figure(figsize=(10,5))
sns.boxplot(data=df[['Irrigation_Recommended(mm)','Soil_Moisture(%)']])
plt.title('Box plot for Soil moisture and Recommended Irrigation')
plt.show()

'''

'\n#1.SOIL MOISTURE LINE GRAPH\nplt.figure(figsize=(10,5))\nsns.lineplot(x=df[\'Date\'], y=df[\'Soil_Moisture(%)\'])\nplt.title(\'Soil Moisture Trend Over Time\')\nplt.xlabel(\'Date\')\nplt.ylabel(\'Soil Moisture(%)\')\nplt.show()\n\n#2.SOIL MOISTURE HISTOGRAM\nplt.figure(figsize=(10,5))\nsns.histplot(df[\'Soil_Moisture(%)\'],bins=30, kde=True)\nplt.title(\'Soil Moisture Histogram\')\nplt.show()\n\n#3.MOISTURE VS TEMPERATURE SCATTER PLOT\nplt.figure(figsize=(10,5))\nsns.scatterplot(x=df[\'Soil_Moisture(%)\'], y=df[\'Temperature(C)\'])\nplt.title(\'Soil Moisture vs Temperature\')\nplt.xlabel(\'Temperature\')\nplt.ylabel(\'Soil Moisture(%)\')\nplt.show()\n\n#4.BOXPLOT FOR MOISTURE AND TEMPEATURE\nplt.figure(figsize=(10,5))\nsns.boxplot(data=df[[\'Temperature(C)\',\'Soil_Moisture(%)\']])\nplt.title(\'Box plot for Soil moisture and Temperature\')\nplt.show()\n\n#5.HEATMAP \nplt.figure(figsize=(8,5))\n#COMPUTE CORRELATION MATRIX\ncorr_matrix = df[[\'Soil_Moisture(%)\',\'Soil_pH\',\'Temperat

In [12]:
#Extracting features from Date column
df['Month'] = df['Date'].dt.month
df['Day'] = df['Date'].dt.day
df['Day_of_week'] = df['Date'].dt.dayofweek
df['Day_of_year'] = df['Date'].dt.dayofyear
df['Quarter'] = df['Date'].dt.quarter

#Cyclical encoding for month and day to capture seasonality
df["Month_sin"]      = np.sin(2 * np.pi * df["Month"] / 12)
df["Month_cos"]      = np.cos(2 * np.pi * df["Month"] / 12)
df["Day_of_year_sin"]  = np.sin(2 * np.pi * df["Day_of_year"] / 365)
df["Day_of_year_cos"]  = np.cos(2 * np.pi * df["Day_of_year"] / 365)

#TEMPORAL SPLITTING
train_size = int(len(df) * 0.8)
train_df = df.iloc[:train_size].copy()
test_df = df.iloc[train_size:].copy()

In [13]:
#FEATURE ENGINEERING

#Interaction/Agronomic features
train_df['Monthly_temp'] = train_df['Temperature(C)'] * train_df['Month']
test_df['Monthly_temp'] = test_df['Temperature(C)'] * test_df['Month']

train_df["Temp_x_pH"]         = train_df["Temperature(C)"] * train_df["Soil_pH"]
test_df["Temp_x_pH"]         = test_df["Temperature(C)"] * test_df["Soil_pH"]

train_df["Fert_per_Irrig"]    = train_df["Fertilizer_Recommended(kg/ha)"] / ( train_df["Irrigation_Recommended(mm)"] + 1e-6)
test_df["Fert_per_Irrig"]    = test_df["Fertilizer_Recommended(kg/ha)"] / ( test_df["Irrigation_Recommended(mm)"] + 1e-6)

train_df["Temp_x_Humidity"]   = train_df["Temperature(C)"] * train_df["Humidity(%)"]
test_df["Temp_x_Humidity"]   = test_df["Temperature(C)"] * test_df["Humidity(%)"]

#LAG FEATURES
train_df["Moisture_Lag1"] = train_df["Soil_Moisture(%)"].shift(1)
train_df["Moisture_Lag2"] = train_df["Soil_Moisture(%)"].shift(2)
#df = df.dropna(subset=["Moisture_Lag2"])

test_df["Moisture_Lag2"] = test_df["Soil_Moisture(%)"].shift(1)
test_df["Moisture_Lag2"] = test_df["Soil_Moisture(%)"].shift(2)
# Smooth the target
#df["Moisture_Smoothed"] = df["Soil_Moisture(%)"].rolling(window=3, min_periods=1).mean()

#ROLLING FEATURES
train_df["Moisture_RollingMean3"] = train_df["Soil_Moisture(%)"].rolling(window=3).mean()
test_df["Moisture_RollingMean3"] = test_df["Soil_Moisture(%)"].rolling(window=3).mean()

last_train_values = train_df['Soil_Moisture(%)'].iloc[-3:].values
test_extended = np.concatenate([last_train_values, test_df['Soil_Moisture(%)'].values])
test_df['Moisture_RollingMean3'] = pd.Series(test_extended).rolling(3).mean().iloc[3:].values

#Binned features
train_df["Temp_Bin"]     = pd.cut(train_df["Temperature(C)"], bins=[18,22,27,32,36], labels=["Cool","Mild","Warm","Hot"])
test_df["Temp_Bin"]     = pd.cut(test_df["Temperature(C)"], bins=[18,22,27,32,36], labels=["Cool","Mild","Warm","Hot"])
train_df["pH_Category"]  = pd.cut(train_df["Soil_pH"], bins=[4.0, 5.5, 6.5, 8.1], labels=["Acidic","Neutral","Alkaline"])
test_df["pH_Category"]  = pd.cut(test_df["Soil_pH"], bins=[4.0, 5.5, 6.5, 8.1], labels=["Acidic","Neutral","Alkaline"])

#ENCODING CATEGORICAL FEATURES
train_df = pd.get_dummies(train_df, columns=['Crop_Type'], drop_first=True)
test_df = pd.get_dummies(test_df, columns=['Crop_Type'], drop_first=True)

train_df = pd.get_dummies(train_df, columns=['Temp_Bin'], drop_first=True)
test_df = pd.get_dummies(test_df, columns=['Temp_Bin'], drop_first=True)

train_df = pd.get_dummies(train_df, columns=['pH_Category'], drop_first=True)
test_df = pd.get_dummies(test_df, columns=['pH_Category'], drop_first=True)

#DROPPING NaNs 

train_df = train_df.dropna()
test_df = test_df.dropna()

#ALIGNING COLUMNS
train_df, test_df = train_df.align(test_df, join='left', axis=1, fill_value=0)
print(train_df.columns)


Index(['Sensor_ID', 'Date', 'Soil_Moisture(%)', 'Soil_pH', 'Temperature(C)',
       'Humidity(%)', 'Fertilizer_Recommended(kg/ha)',
       'Irrigation_Recommended(mm)', 'Drone_Image_ID', 'Month', 'Day',
       'Day_of_week', 'Day_of_year', 'Quarter', 'Month_sin', 'Month_cos',
       'Day_of_year_sin', 'Day_of_year_cos', 'Monthly_temp', 'Temp_x_pH',
       'Fert_per_Irrig', 'Temp_x_Humidity', 'Moisture_Lag1', 'Moisture_Lag2',
       'Moisture_RollingMean3', 'Crop_Type_Lettuce', 'Crop_Type_Maize',
       'Crop_Type_Tomatoes', 'Crop_Type_Wheat', 'Temp_Bin_Mild',
       'Temp_Bin_Warm', 'Temp_Bin_Hot', 'pH_Category_Neutral',
       'pH_Category_Alkaline'],
      dtype='object')


In [14]:
features = ['Soil_Moisture(%)', 'Soil_pH', 'Temperature(C)',
       'Humidity(%)', 'Fertilizer_Recommended(kg/ha)',
       'Irrigation_Recommended(mm)', 'Month_sin', 'Month_cos',
       'Day_of_year_sin', 'Day_of_year_cos', 'Monthly_temp', 'Temp_x_pH',
       'Fert_per_Irrig', 'Temp_x_Humidity', 'Moisture_Lag1', 'Moisture_Lag2',
       'Moisture_RollingMean3', 'Crop_Type_Lettuce', 'Crop_Type_Maize',
       'Crop_Type_Tomatoes', 'Crop_Type_Wheat', 'Temp_Bin_Mild',
       'Temp_Bin_Warm', 'Temp_Bin_Hot', 'pH_Category_Neutral',
       'pH_Category_Alkaline']

#SCALING FEATURES
scaler = MinMaxScaler()

trained_scale = scaler.fit_transform(train_df[features])
test_scaled = scaler.transform(test_df[features])

#SEQUENCE WINDOWING
def create_sequences(data, target_index, window_size=24):
    X, y = [], []
    
    for i in range(len(data) - window_size):
        X.append(data[i:i+window_size])
        y.append(data[i+window_size, target_index])
    
    return np.array(X), np.array(y)

In [15]:
target = 'Soil_Moisture(%)'
target_index = features.index(target)

X_train, y_train = create_sequences(trained_scale, target_index)
X_test, y_test = create_sequences(test_scaled, target_index)

In [16]:
#FLATTENING FOR MODELS THAT USE TABULAR DATA
X_train_flat = X_train.reshape(X_train.shape[0], -1)
X_test_flat = X_test.reshape(X_test.shape[0], -1)

In [17]:
#RIDGE REGRESSION TRAINING
ridge_model = Ridge(alpha=1.0)
ridge_model.fit(X_train_flat, y_train)
y_pred_ridge = ridge_model.predict(X_test_flat)

In [18]:
#RANDOM FOREST TRAINING
rf_model = RandomForestRegressor(
    n_estimators=100,
    max_depth=10,
    random_state=42
)
rf_model.fit(X_train_flat, y_train)

y_pred_rf = rf_model.predict(X_test_flat)

In [19]:
#GRADIENT BOOSTING TRAINING
gb_model = GradientBoostingRegressor()
gb_model.fit(X_train_flat, y_train)

y_pred_gb = gb_model.predict(X_test_flat)

In [20]:
#LSTM TRAINING
lstm_model = Sequential([
    LSTM(64, input_shape=(X_train.shape[1], X_train.shape[2])),
    Dense(32, activation='relu'),
    Dense(1)
])

lstm_model.compile(optimizer='adam', loss='mse')

lstm_model.fit(
    X_train, y_train,
    epochs=50,          
    batch_size=32,
    validation_data=(X_test, y_test)
)

y_pred_lstm = lstm_model.predict(X_test)

c:\Users\hi\AppData\Local\Programs\Python\Python311\Lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Epoch 1/50
20/20 ━━━━━━━━━━━━━━━━━━━━ 16s 231ms/step - loss: 0.1307 - val_loss: 0.0823
Epoch 2/50
20/20 ━━━━━━━━━━━━━━━━━━━━ 2s 83ms/step - loss: 0.0909 - val_loss: 0.0821
Epoch 3/50
20/20 ━━━━━━━━━━━━━━━━━━━━ 2s 85ms/step - loss: 0.0883 - val_loss: 0.0838
Epoch 4/50
20/20 ━━━━━━━━━━━━━━━━━━━━ 3s 169ms/step - loss: 0.0873 - val_loss: 0.0803
Epoch 5/50
20/20 ━━━━━━━━━━━━━━━━━━━━ 6s 186ms/step - loss: 0.0863 - val_loss: 0.0864
Epoch 6/50
20/20 ━━━━━━━━━━━━━━━━━━━━ 4s 87ms/step - loss: 0.0873 - val_loss: 0.0866
Epoch 7/50
20/20 ━━━━━━━━━━━━━━━━━━━━ 6s 309ms/step - loss: 0.0883 - val_loss: 0.0820
Epoch 8/50
20/20 ━━━━━━━━━━━━━━━━━━━━ 17s 561ms/step - loss: 0.0854 - val_loss: 0.0807
Epoch 9/50
20/20 ━━━━━━━━━━━━━━━━━━━━ 6s 105ms/step - loss: 0.0868 - val_loss: 0.0808
Epoch 10/50
20/20 ━━━━━━━━━━━━━━━━━━━━ 3s 99ms/step - loss: 0.0857 - val_loss: 0.0844
Epoch 11/50
20/20 ━━━━━━━━━━━━━━━━━━━━ 2s 100ms/step - loss: 0.0850 - val_loss: 0.0822
Epoch 12/50
20/20 ━━━━━━━━━━━━━━━━━━━━ 2s 85ms/step - 

In [21]:
#EVALUATION
def evaluate(y_true, y_pred, model):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)
    print(f"{model} Results:")
    print("MAE:", mae)
    print("RMSE:", rmse)
    print("r2:", r2) 
evaluate(y_test, y_pred_ridge, "Ridge")
evaluate(y_test, y_pred_rf, "Random Forest")
evaluate(y_test, y_pred_gb, "Gradient Boost")
evaluate(y_test, y_pred_lstm, "LSTM")

Ridge Results:
MAE: 0.35343002791911204
RMSE: 0.44115734191247613
r2: -1.423024776274632
Random Forest Results:
MAE: 0.2489350136450606
RMSE: 0.28688998078634037
r2: -0.024711464545202677
Gradient Boost Results:
MAE: 0.2542378066924069
RMSE: 0.2987363945446932
r2: -0.11108453482513525
LSTM Results:
MAE: 0.2667239796696617
RMSE: 0.3215796086991596
r2: -0.28750177714265646


In [22]:
#MODEL IMPROVEMENT
from sklearn.model_selection import TimeSeriesSplit,GridSearchCV

tscv = TimeSeriesSplit(n_splits=5)

In [23]:
#RANDOM FOREST TUNING
rf_params = {
    'n_estimators': [100, 200],
    'max_depth': [5, 10, 20],
    'min_samples_split': [2, 5],
    'min_samples_leaf': [1, 2]
}

rf = RandomForestRegressor(random_state=42)

rf_grid = GridSearchCV(
    estimator=rf,
    param_grid=rf_params,
    cv=tscv,
    scoring='neg_mean_squared_error',
    n_jobs=-1,
    verbose=2
)

rf_grid.fit(X_train_flat, y_train)

tuned_rf = rf_grid.best_estimator_

print("best rf parameters:", rf_grid.best_params_)

y_pred_rf = tuned_rf.predict(X_test_flat)

Fitting 5 folds for each of 24 candidates, totalling 120 fits
best rf parameters: {'max_depth': 5, 'min_samples_leaf': 2, 'min_samples_split': 5, 'n_estimators': 200}


In [24]:
#GRADIENT BOOSTING TUNING
gb_params = {
    'n_estimators': [100, 200],
    'learning_rate': [0.01, 0.1],
    'max_depth': [3, 5],
    'subsample': [0.8, 1.0]
}

gb = GradientBoostingRegressor(random_state=42)

gb_grid = GridSearchCV(
    estimator=gb,
    param_grid=gb_params,
    cv=tscv,
    scoring='neg_mean_squared_error',
    n_jobs=-1,
    verbose=2
)

gb_grid.fit(X_train_flat, y_train)

tuned_gb = gb_grid.best_estimator_
print("best gb parameters", gb_grid.best_params_)

y_pred_gb = tuned_gb.predict(X_test_flat)

Fitting 5 folds for each of 16 candidates, totalling 80 fits
best gb parameters {'learning_rate': 0.01, 'max_depth': 3, 'n_estimators': 100, 'subsample': 0.8}


In [25]:
#EVALUATION OF THE TWO MODELS
def evaluation(y_true, y_pred, name):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)
    print(f"{name} Results")
    print("MAE:", mae)
    print("RMSE:", rmse)
    print("r2", r2)
    

evaluation(y_test, y_pred_rf, "Tuned Random Forest")
evaluation(y_test, y_pred_gb, "Tuned Gradient Boosting")

Tuned Random Forest Results
MAE: 0.24696522014860353
RMSE: 0.28657776565236043
r2 -0.022482342843511027
Tuned Gradient Boosting Results
MAE: 0.24454756428245458
RMSE: 0.2841010391021605
r2 -0.0048852617646071295
